# 面试问题：LLM KV Cache 为什么适合非对称量化，怎样检测离群值并回退到高精度？

        ## 可直接复述的回答主线

        1. KV Cache 随序列长度线性增长，低比特存储能降低显存和带宽，但量化误差会直接改变 attention 输出。
2. 全张量对称量化用单个 max-abs 定标，一个离群值就会放大量化步长并抹掉多数正常值。
3. 更实用的方案按 K/V、按 head 计算非对称 scale 和 zero-point，以覆盖偏移分布。
4. 先用稳健阈值识别离群值，正常值量化为 int4，离群值及其位置单独以高精度保存并在反量化后覆盖。
5. 评测不能只看 K/V 元素误差，还要在同一批 query 上比较 attention 输出误差、显存估算和离群回退率。
6. 生产实现还需真实 packed int4 kernel、分组策略、prefill/decode 分离、校准集、延迟评测和模型层级回退。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例构造五个不同长度请求，每个请求有 2 个 attention head、head_dim=4；其中三条注入不同强度的 K/V 离群值。我们在同一份 KV 和 query 上比较全张量对称 int4 与按 head 非对称 int4 加 FP16 离群回退。

In [1]:
import math  # 计算 scaled dot-product attention 的维度缩放。
import numpy as np  # 使用基础数组手写量化、反量化和 attention。
rng = np.random.default_rng(29)  # 固定随机数以保证五个请求和保存输出可复现。
request_specs = [{"id": "kv-01", "tokens": 8, "k_outlier": 0.0, "v_outlier": 0.0}, {"id": "kv-02", "tokens": 10, "k_outlier": 5.0, "v_outlier": 0.0}, {"id": "kv-03", "tokens": 12, "k_outlier": 0.0, "v_outlier": -6.0}, {"id": "kv-04", "tokens": 9, "k_outlier": 8.0, "v_outlier": 5.0}, {"id": "kv-05", "tokens": 11, "k_outlier": -4.5, "v_outlier": 3.5}]  # 定义五个具有不同长度和离群模式的请求。
requests = []  # 保存每个请求的 query、K 和 V 张量。
for index, spec in enumerate(request_specs):  # 按请求规格生成确定性的 KV Cache。
    keys = rng.normal(loc=0.05 * (index % 2), scale=0.28, size=(spec["tokens"], 2, 4)).astype(np.float64)  # 生成带轻微 head 偏移的 K。
    values = rng.normal(loc=-0.04 * (index % 3), scale=0.32, size=(spec["tokens"], 2, 4)).astype(np.float64)  # 生成带轻微非对称分布的 V。
    query = rng.normal(loc=0.0, scale=0.45, size=(2, 4)).astype(np.float64)  # 为当前请求生成两个 head 的 query。
    if spec["k_outlier"] != 0.0:  # 检查是否需要向 K 注入离群值。
        keys[index % spec["tokens"], index % 2, index % 4] = spec["k_outlier"]  # 在确定位置注入高幅 K 元素。
    if spec["v_outlier"] != 0.0:  # 检查是否需要向 V 注入离群值。
        values[(index + 2) % spec["tokens"], (index + 1) % 2, (index + 1) % 4] = spec["v_outlier"]  # 在另一位置注入高幅 V 元素。
    requests.append({"id": spec["id"], "query": query, "keys": keys, "values": values, "spec": spec})  # 保存一条完整 attention 输入。
print("教学实验输入：五条请求的 KV Cache")  # 标记下方为数值量化案例输入。
print("请求      K shape       V shape       K范围                 V范围                 注入离群")  # 输出输入预览表头。
for request in requests:  # 逐条展示张量大小和真实数值范围。
    keys = request["keys"]  # 读取当前请求 K。
    values = request["values"]  # 读取当前请求 V。
    spec = request["spec"]  # 读取注入离群设置。
    print(f"{request['id']:<9} {str(keys.shape):<13} {str(values.shape):<13} [{keys.min():>6.2f},{keys.max():>6.2f}]   [{values.min():>6.2f},{values.max():>6.2f}]   K={spec['k_outlier']},V={spec['v_outlier']}")  # 输出当前请求分布和离群强度。

教学实验输入：五条请求的 KV Cache
请求      K shape       V shape       K范围                 V范围                 注入离群
kv-01     (8, 2, 4)     (8, 2, 4)     [ -0.63,  0.84]   [ -0.84,  0.70]   K=0.0,V=0.0
kv-02     (10, 2, 4)    (10, 2, 4)    [ -0.57,  5.00]   [ -0.83,  0.82]   K=5.0,V=0.0
kv-03     (12, 2, 4)    (12, 2, 4)    [ -0.66,  0.78]   [ -6.00,  0.79]   K=0.0,V=-6.0
kv-04     (9, 2, 4)     (9, 2, 4)     [ -0.57,  8.00]   [ -1.02,  5.00]   K=8.0,V=5.0
kv-05     (11, 2, 4)    (11, 2, 4)    [ -4.50,  0.60]   [ -0.57,  3.50]   K=-4.5,V=3.5


## 2. Baseline / 基线：全张量一个 scale 的对称 int4

int4 有符号范围取 -7 到 7。整个 K 或 V 共用 `max(abs(x))/7`，离群值会把正常 0.2～0.4 范围压到很少几个整数桶。

In [2]:
def symmetric_int4(tensor):  # 手写全张量对称 int4 量化和反量化。
    maximum = float(np.max(np.abs(tensor)))  # 用最大绝对值覆盖整个张量范围。
    scale = maximum / 7.0 if maximum > 1.0e-12 else 1.0  # 把最大值映射到 int4 正端并保护全零张量。
    quantized = np.clip(np.rint(tensor / scale), -7, 7).astype(np.int8)  # 舍入并裁剪到对称 int4 逻辑范围。
    restored = quantized.astype(np.float64) * scale  # 用同一个 scale 反量化回浮点数。
    return restored, {"scale": scale, "minimum_code": int(quantized.min()), "maximum_code": int(quantized.max())}  # 返回重构张量和量化元数据。
def attention_output(query, keys, values):  # 手写多头 scaled dot-product attention 前向计算。
    scores = np.einsum("hd,thd->ht", query, keys) / math.sqrt(query.shape[-1])  # 计算每个 head 对各 Token 的缩放点积。
    shifted = scores - scores.max(axis=1, keepdims=True)  # 减去每头最大值避免 softmax 溢出。
    weights = np.exp(shifted)  # 对稳定 logits 取指数。
    weights = weights / weights.sum(axis=1, keepdims=True)  # 逐 head 归一化注意力权重。
    output = np.einsum("ht,thd->hd", weights, values)  # 用注意力权重加权求和 V。
    return output, scores, weights  # 返回输出以及可解释的 logits 和权重。
baseline_rows = []  # 保存五个请求的基线 attention 误差。
for request in requests:  # 对同一批原始 K/V 执行全张量量化。
    reference, _, _ = attention_output(request["query"], request["keys"], request["values"])  # 计算未量化 FP64 参考输出。
    restored_keys, key_meta = symmetric_int4(request["keys"])  # 对整个 K 使用一个对称 scale。
    restored_values, value_meta = symmetric_int4(request["values"])  # 对整个 V 使用一个对称 scale。
    approximate, _, _ = attention_output(request["query"], restored_keys, restored_values)  # 用反量化 KV 计算近似输出。
    error = float(np.mean(np.abs(approximate - reference)))  # 计算 attention 输出平均绝对误差。
    baseline_rows.append({"id": request["id"], "error": error, "key_scale": key_meta["scale"], "value_scale": value_meta["scale"], "output": approximate, "reference": reference})  # 保存基线误差和 scale。
print("Baseline 全张量对称 int4")  # 标记下表展示离群值引起的粗量化步长。
print("请求      K scale    V scale    attention MAE")  # 输出基线量化结果表头。
for row in baseline_rows:  # 逐请求展示 scale 和最终输出误差。
    print(f"{row['id']:<9} {row['key_scale']:>8.4f} {row['value_scale']:>10.4f} {row['error']:>14.6f}")  # 输出当前请求的端到端误差。

Baseline 全张量对称 int4
请求      K scale    V scale    attention MAE
kv-01       0.1193     0.1207       0.008185
kv-02       0.7143     0.1184       0.014879
kv-03       0.1116     0.8571       0.084370
kv-04       1.1429     0.7143       0.054205
kv-05       0.6429     0.5000       0.035907


## 3. 底层实现：按 head 非对称 int4 与离群回退

每个 head 用中位绝对值构造稳健阈值；正常值用 `[min,max]` 计算 15 个间隔的 scale 和 zero-point，离群值的位置与原值单独保存，反量化后精确覆盖。

In [3]:
def asymmetric_int4_with_outliers(tensor, threshold_multiplier=3.5, absolute_floor=0.75):  # 手写按 head 的非对称量化与离群回退。
    restored = np.empty_like(tensor, dtype=np.float64)  # 分配最终反量化张量。
    metadata = []  # 保存每个 head 的 scale、zero-point 和离群位置。
    for head in range(tensor.shape[1]):  # 独立处理每个 attention head。
        original = tensor[:, head, :].reshape(-1).astype(np.float64)  # 展平当前 head 的 Token 和维度。
        median_absolute = float(np.median(np.abs(original)))  # 用中位绝对值估计正常幅度。
        threshold = max(absolute_floor, threshold_multiplier * median_absolute)  # 结合绝对下限得到稳健离群阈值。
        outlier_mask = np.abs(original) > threshold  # 标记需要高精度回退的元素。
        normal = original[~outlier_mask]  # 只用正常值校准 int4 范围。
        lower = float(normal.min()) if normal.size else 0.0  # 读取正常值最小值并保护全离群情况。
        upper = float(normal.max()) if normal.size else 0.0  # 读取正常值最大值并保护全离群情况。
        constant_normal = upper - lower <= 1.0e-12  # 识别无法用 min-max 计算 scale 的常量正常值集合。
        scale = (upper - lower) / 15.0 if not constant_normal else 1.0  # 把非对称范围映射到十五个间隔并为常量路径提供有限占位 scale。
        zero_point = int(np.clip(np.rint(-8.0 - lower / scale), -8, 7))  # 计算使 lower 接近 -8 的有符号 zero-point。
        quantized = np.clip(np.rint(original / scale + zero_point), -8, 7).astype(np.int8)  # 将当前 head 编码到有符号 int4 逻辑范围。
        head_restored = (quantized.astype(np.float64) - zero_point) * scale  # 使用 scale 和 zero-point 反量化正常路径。
        head_restored[~outlier_mask] = lower if constant_normal else head_restored[~outlier_mask]  # 对常量正常值显式恢复其唯一浮点值。
        head_restored[outlier_mask] = original[outlier_mask]  # 用高精度离群值覆盖被裁剪的位置。
        restored[:, head, :] = head_restored.reshape(tensor.shape[0], tensor.shape[2])  # 把当前 head 写回原始布局。
        metadata.append({"head": head, "scale": scale, "zero_point": zero_point, "constant_value": lower if constant_normal else None, "threshold": threshold, "outlier_indices": np.flatnonzero(outlier_mask).tolist(), "outlier_values": original[outlier_mask].tolist()})  # 保存可观测量化参数、常量回退和离群内容。
    return restored, metadata  # 返回按 head 重构张量和全部元数据。
preview_request = requests[3]  # 选择同时含 K/V 离群值的 kv-04 展示中间量。
preview_keys, preview_key_meta = asymmetric_int4_with_outliers(preview_request["keys"])  # 量化并还原 kv-04 的 K。
preview_values, preview_value_meta = asymmetric_int4_with_outliers(preview_request["values"])  # 量化并还原 kv-04 的 V。
print("kv-04 按 head 量化中间量")  # 标记下表展示非对称参数和离群位置。
print("tensor head   scale   zero_point  threshold  outlier_indices  outlier_values")  # 输出中间量表头。
for tensor_name, rows in (("K", preview_key_meta), ("V", preview_value_meta)):  # 依次展示 K 和 V 的每头参数。
    for row in rows:  # 遍历当前张量的两个 head。
        print(f"{tensor_name:<6} {row['head']:>4} {row['scale']:>8.4f} {row['zero_point']:>11} {row['threshold']:>10.4f} {str(row['outlier_indices']):<17} {row['outlier_values']}")  # 输出当前 head 的量化和回退证据。
preview_reference, preview_scores, preview_weights = attention_output(preview_request["query"], preview_request["keys"], preview_request["values"])  # 计算 kv-04 原始 attention 中间量。
preview_approximate, preview_quantized_scores, preview_quantized_weights = attention_output(preview_request["query"], preview_keys, preview_values)  # 计算 kv-04 量化后 attention 中间量。
print("kv-04 head0 原始weights=", np.round(preview_weights[0], 4).tolist())  # 展示原始 attention 权重而不只打印 shape。
print("kv-04 head0 修正weights=", np.round(preview_quantized_weights[0], 4).tolist())  # 展示量化后 attention 权重变化。

kv-04 按 head 量化中间量
tensor head   scale   zero_point  threshold  outlier_indices  outlier_values
K         0   0.0663          -2     0.7500 []                []
K         1   0.0755           0     0.7500 [15]              [8.0]
V         0   0.0826          -1     0.7500 [9, 20]           [-1.021589881295829, 5.0]
V         1   0.0909          -1     0.7500 []                []
kv-04 head0 原始weights= [0.1203, 0.1026, 0.1199, 0.1269, 0.1175, 0.0755, 0.1311, 0.1093, 0.0969]
kv-04 head0 修正weights= [0.1232, 0.1023, 0.1205, 0.1272, 0.1167, 0.0746, 0.1295, 0.1092, 0.0967]


## 4. 逐请求结果与结果解读

两种方案都在相同 query、K、V 上计算 attention。内存是逻辑估算：正常元素按 4 bit，离群值按 FP16 加二维位置索引；这里没有假装 NumPy 的 `int8` 已经真实 packed。

In [4]:
corrected_rows = []  # 保存按 head 非对称量化的逐请求结果。
for request, baseline in zip(requests, baseline_rows):  # 对每条请求与对应基线进行同数据比较。
    restored_keys, key_metadata = asymmetric_int4_with_outliers(request["keys"])  # 量化并还原当前 K。
    restored_values, value_metadata = asymmetric_int4_with_outliers(request["values"])  # 量化并还原当前 V。
    approximate, _, _ = attention_output(request["query"], restored_keys, restored_values)  # 计算修正方案 attention 输出。
    error = float(np.mean(np.abs(approximate - baseline["reference"])))  # 计算相对未量化参考的平均绝对误差。
    outlier_count = sum(len(row["outlier_indices"]) for row in key_metadata + value_metadata)  # 统计 K/V 合计高精度回退元素。
    element_count = request["keys"].size + request["values"].size  # 统计 KV Cache 元素总数。
    fp16_bytes = element_count * 2.0  # 估算未量化 FP16 缓存字节数。
    logical_bytes = element_count * 0.5 + outlier_count * 6.0  # 估算 packed int4 加每个离群 FP16 值和四字节位置的字节数。
    corrected_rows.append({"id": request["id"], "baseline_error": baseline["error"], "error": error, "outliers": outlier_count, "fp16_bytes": fp16_bytes, "logical_bytes": logical_bytes, "compression": fp16_bytes / logical_bytes})  # 保存逐请求误差和逻辑显存指标。
baseline_mean_error = float(np.mean([row["error"] for row in baseline_rows]))  # 计算全张量对称方案平均 attention 误差。
corrected_mean_error = float(np.mean([row["error"] for row in corrected_rows]))  # 计算非对称离群回退方案平均 attention 误差。
print("请求      对称int4_MAE  非对称回退_MAE  outliers  FP16字节  逻辑字节  压缩比")  # 输出同数据逐请求对照表头。
for row in corrected_rows:  # 逐请求展示质量与内存权衡。
    print(f"{row['id']:<9} {row['baseline_error']:>13.6f} {row['error']:>15.6f} {row['outliers']:>9} {row['fp16_bytes']:>9.0f} {row['logical_bytes']:>9.1f} {row['compression']:>7.2f}x")  # 输出当前请求的端到端结果。
print(f"结果解读：平均attention MAE从{baseline_mean_error:.6f}降到{corrected_mean_error:.6f}；回退率越高，质量更稳但压缩收益会下降。")  # 解释精度与显存之间的真实权衡。

请求      对称int4_MAE  非对称回退_MAE  outliers  FP16字节  逻辑字节  压缩比
kv-01          0.008185        0.002783         3       256      82.0    3.12x
kv-02          0.014879        0.005920         3       320      98.0    3.27x
kv-03          0.084370        0.005926         3       384     114.0    3.37x
kv-04          0.054205        0.005637         3       288      90.0    3.20x
kv-05          0.035907        0.007787         2       352     100.0    3.52x
结果解读：平均attention MAE从0.039509降到0.005611；回退率越高，质量更稳但压缩收益会下降。


## 5. 失败案例与修正：常量 head 导致 scale=0

直接用 `(max-min)/15` 量化常量 head 会除以零并产生非有限值。核心函数显式把退化 scale 设为 1，并在元数据中保留唯一常量值供反量化覆盖。

In [5]:
constant_head = np.full((6, 1, 4), 0.25, dtype=np.float64)  # 构造 max 等于 min 的真实退化缓存块。
naive_scale = float((constant_head.max() - constant_head.min()) / 15.0)  # 朴素非对称校准得到零 scale。
with np.errstate(divide="ignore", invalid="ignore"):  # 暂时屏蔽预期的除零告警以展示失败数值。
    naive_codes = np.rint(constant_head / naive_scale)  # 用零 scale 量化并产生无穷或非数值。
naive_has_nonfinite = not bool(np.isfinite(naive_codes).all())  # 检查失败实现是否出现非有限编码。
safe_constant, safe_metadata = asymmetric_int4_with_outliers(constant_head)  # 用带退化保护的核心实现处理常量 head。
safe_is_finite = bool(np.isfinite(safe_constant).all())  # 检查修正结果是否全部有限。
safe_constant_error = float(np.max(np.abs(safe_constant - constant_head)))  # 计算常量 head 最大重构误差。
print(f"错误行为：naive_scale={naive_scale}，nonfinite={naive_has_nonfinite}，codes_sample={naive_codes.reshape(-1)[:3].tolist()}")  # 展示除零导致的失败输出。
print(f"修正行为：safe_scale={safe_metadata[0]['scale']}，finite={safe_is_finite}，max_error={safe_constant_error:.6f}")  # 展示 scale 保护后的有限结果。

错误行为：naive_scale=0.0，nonfinite=True，codes_sample=[inf, inf, inf]
修正行为：safe_scale=1.0，finite=True，max_error=0.000000


## 6. 生产边界

NumPy 的 `int8` 容器不是 packed int4，本例内存只是逻辑估算。线上需要 GPU packing/depacking kernel、每层和每 head 校准、旋转位置编码后的分布测试、prefill/decode 吞吐、长上下文质量、动态离群预算和 FP16 回退开关。

In [6]:
total_outliers = sum(row["outliers"] for row in corrected_rows)  # 汇总五个请求的高精度离群元素数。
total_elements = sum(request["keys"].size + request["values"].size for request in requests)  # 汇总全部 KV 元素数。
diagnostics = {"requests": len(requests), "baseline_attention_mae": baseline_mean_error, "corrected_attention_mae": corrected_mean_error, "outlier_rate": total_outliers / total_elements, "mean_logical_compression": float(np.mean([row["compression"] for row in corrected_rows])), "real_int4_packing": False}  # 汇总精度、回退率和实现边界。
print("生产监控快照：", diagnostics)  # 输出 KV 量化上线前需要关注的指标。

生产监控快照： {'requests': 5, 'baseline_attention_mae': 0.039509336946076425, 'corrected_attention_mae': 0.005610685549324189, 'outlier_rate': 0.0175, 'mean_logical_compression': 3.2951356789185504, 'real_int4_packing': False}


## 7. 最小回归测试

断言覆盖样本规模、端到端误差改善、离群检测、常量失败修复和有限输出。

In [7]:
assert len(requests) >= 5 and all(request["keys"].shape[1:] == (2, 4) for request in requests)  # 保证至少五条多头 KV 请求参与评测。
assert corrected_mean_error < baseline_mean_error  # 保证同一批请求的平均 attention 输出误差确实下降。
assert sum(row["error"] <= row["baseline_error"] for row in corrected_rows) >= 4  # 保证改善不是只来自单个偶然请求。
assert any(row["outliers"] > 0 for row in corrected_rows) and total_outliers < total_elements  # 保证离群回退真实发生且没有把全部元素回退。
assert naive_has_nonfinite and safe_is_finite and safe_constant_error < 1.0e-12  # 保证 scale=0 失败真实复现并被安全修复。
assert all(np.isfinite(row["error"]) and row["logical_bytes"] < row["fp16_bytes"] for row in corrected_rows)  # 保证所有误差有限且逻辑缓存仍小于 FP16。